In [2]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

print("=== Task 1: Card Probabilities ===")
print(f"P(Red) = {26/52:.4f}")
print(f"P(Heart | Red) = {13/26:.4f}")
print(f"P(Diamond | Face) = {3/12:.4f}")
print(f"P(Spade or Queen | Face) = {(3+1)/12:.4f}")

/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in a future release. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


=== Task 1: Card Probabilities ===
P(Red) = 0.5000
P(Heart | Red) = 0.5000
P(Diamond | Face) = 0.2500
P(Spade or Queen | Face) = 0.3333


In [3]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([
    ('Intelligence', 'Grade'),
    ('StudyHours', 'Grade'),
    ('Difficulty', 'Grade'),
    ('Grade', 'Pass')
])

cpd_i = TabularCPD('Intelligence', 2, [[0.3], [0.7]])
cpd_s = TabularCPD('StudyHours', 2, [[0.4], [0.6]])
cpd_d = TabularCPD('Difficulty', 2, [[0.4], [0.6]])

cpd_g = TabularCPD(
    variable='Grade', variable_card=3,
    values=[
        [0.05, 0.10, 0.15, 0.20, 0.25, 0.35, 0.40, 0.50],
        [0.25, 0.30, 0.35, 0.40, 0.45, 0.40, 0.40, 0.35],
        [0.70, 0.60, 0.50, 0.40, 0.30, 0.25, 0.20, 0.15]
    ],
    evidence=['Intelligence', 'StudyHours', 'Difficulty'],
    evidence_card=[2, 2, 2]
)

cpd_p = TabularCPD(
    variable='Pass', variable_card=2,
    values=[
        [0.05, 0.20, 0.50],
        [0.95, 0.80, 0.50]
    ],
    evidence=['Grade'],
    evidence_card=[3]
)

model.add_cpds(cpd_i, cpd_s, cpd_d, cpd_g, cpd_p)
assert model.check_model()

infer = VariableElimination(model)

r1 = infer.query(['Pass'], evidence={'StudyHours': 1, 'Difficulty': 0})
print("\n=== Task 2 ===")
print("P(Pass | StudyHours=Sufficient, Difficulty=Hard):")
print(r1)

r2 = infer.query(['Intelligence'], evidence={'Pass': 1})
print("P(Intelligence | Pass=Yes):")
print(r2)


=== Task 2 ===
P(Pass | StudyHours=Sufficient, Difficulty=Hard):
+---------+-------------+
| Pass    |   phi(Pass) |
+=========+=============+
| Pass(0) |      0.2383 |
+---------+-------------+
| Pass(1) |      0.7618 |
+---------+-------------+
P(Intelligence | Pass=Yes):
+-----------------+---------------------+
| Intelligence    |   phi(Intelligence) |
+=================+=====================+
| Intelligence(0) |              0.2634 |
+-----------------+---------------------+
| Intelligence(1) |              0.7366 |
+-----------------+---------------------+


In [4]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([
    ('Disease', 'Fever'),
    ('Disease', 'Cough'),
    ('Disease', 'Fatigue'),
    ('Disease', 'Chills')
])

cpd_disease = TabularCPD('Disease', 2, [[0.3], [0.7]])

cpd_fever = TabularCPD('Fever', 2,
    [[0.1, 0.5], [0.9, 0.5]],
    evidence=['Disease'], evidence_card=[2])

cpd_cough = TabularCPD('Cough', 2,
    [[0.2, 0.4], [0.8, 0.6]],
    evidence=['Disease'], evidence_card=[2])

cpd_fatigue = TabularCPD('Fatigue', 2,
    [[0.3, 0.7], [0.7, 0.3]],
    evidence=['Disease'], evidence_card=[2])

cpd_chills = TabularCPD('Chills', 2,
    [[0.4, 0.6], [0.6, 0.4]],
    evidence=['Disease'], evidence_card=[2])

model.add_cpds(cpd_disease, cpd_fever, cpd_cough, cpd_fatigue, cpd_chills)
assert model.check_model()

infer = VariableElimination(model)

print("\n=== Task 3 ===")
r1 = infer.query(['Disease'], evidence={'Fever': 1, 'Cough': 1})
print("P(Disease | Fever=Yes, Cough=Yes):")
print(r1)

r2 = infer.query(['Disease'], evidence={'Fever': 1, 'Cough': 1, 'Chills': 1})
print("P(Disease | Fever=Yes, Cough=Yes, Chills=Yes):")
print(r2)

r3 = infer.query(['Fatigue'], evidence={'Disease': 0})
print("P(Fatigue=Yes | Disease=Flu):")
print(r3)


=== Task 3 ===
P(Disease | Fever=Yes, Cough=Yes):
+------------+----------------+
| Disease    |   phi(Disease) |
+============+================+
| Disease(0) |         0.5070 |
+------------+----------------+
| Disease(1) |         0.4930 |
+------------+----------------+
P(Disease | Fever=Yes, Cough=Yes, Chills=Yes):
+------------+----------------+
| Disease    |   phi(Disease) |
+============+================+
| Disease(0) |         0.6067 |
+------------+----------------+
| Disease(1) |         0.3933 |
+------------+----------------+
P(Fatigue=Yes | Disease=Flu):
+------------+----------------+
| Fatigue    |   phi(Fatigue) |
+============+================+
| Fatigue(0) |         0.3000 |
+------------+----------------+
| Fatigue(1) |         0.7000 |
+------------+----------------+


In [6]:
import numpy as np
from itertools import product

states = ["Sunny", "Cloudy", "Rainy"]
state_index = {s: i for i, s in enumerate(states)}

transition_matrix = np.array([
    [0.6, 0.3, 0.1],
    [0.3, 0.4, 0.3],
    [0.2, 0.3, 0.5]
])

def simulate_weather(initial_state, num_steps):
    current = state_index[initial_state]
    sequence = [initial_state]
    for _ in range(num_steps):
        current = np.random.choice(len(states), p=transition_matrix[current])
        sequence.append(states[current])
    return sequence

np.random.seed(42)
sequence = simulate_weather("Sunny", 10)
print("Simulated weather sequence:")
print(" -> ".join(sequence))

def prob_at_least_3_rainy(initial_state, num_steps, min_rainy=3):
    num_simulations = 100000
    count = 0
    start = state_index[initial_state]
    for _ in range(num_simulations):
        current = start
        rainy_days = 0
        for _ in range(num_steps):
            current = np.random.choice(len(states), p=transition_matrix[current])
            if states[current] == "Rainy":
                rainy_days += 1
        if rainy_days >= min_rainy:
            count += 1
    return count / num_simulations

prob = prob_at_least_3_rainy("Sunny", 10)
print(f"\nP(at least 3 rainy days in 10 days | Start=Sunny) ≈ {prob:.4f}")

Simulated weather sequence:
Sunny -> Sunny -> Rainy -> Rainy -> Rainy -> Sunny -> Sunny -> Sunny -> Cloudy -> Cloudy -> Rainy


KeyboardInterrupt: 